# Marketplace Detection System - Getting Started

This notebook demonstrates how to use the marketplace detection system to classify items commonly sold on marketplace platforms.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

from src.pipeline import create_pipeline
from src.tier1_detector import create_category_detector
from src.tier2_detectors.electronics import create_electronics_detector
from PIL import Image
import json

print("✓ Imports successful!")

## 2. Initialize the Pipeline

The pipeline combines:
- **Tier 1**: Category classification (electronics, furniture, clothing, etc.)
- **Tier 2**: Specialized attribute detection (brand, model, condition, etc.)

In [ ]:
# Initialize the complete pipeline
pipeline = create_pipeline(
    confidence_threshold=0.5,
    enable_tier2=True
)

print("\n✓ Pipeline initialized!")

## 3. Check Supported Categories

In [ ]:
categories = pipeline.get_supported_categories()

print("Supported categories:")
print("-" * 60)
for cat, has_tier2 in sorted(categories.items()):
    tier2_status = "✓ Tier 2 available" if has_tier2 else "○ Tier 1 only"
    print(f"{cat:30} {tier2_status}")

## 4. Test with a Single Image

Replace `'path/to/your/image.jpg'` with an actual image path to test.

In [ ]:
# Example: Create a test image
test_img = Image.new('RGB', (300, 300), color='blue')
test_img.save('../data/raw/test_image.jpg')

# Or use your own image
image_path = '../data/raw/test_image.jpg'

# Run prediction
result = pipeline.predict(image_path, return_metadata=True)

# Display results
print(json.dumps(result, indent=2))

## 5. Display Results in Structured Format

In [ ]:
def display_result(result):
    """Display detection results in a readable format."""
    print("=" * 70)
    print("DETECTION RESULTS")
    print("=" * 70)
    
    # Tier 1
    print("\n📊 TIER 1 - Category Detection:")
    print(f"  Category: {result['tier1']['category']}")
    print(f"  Confidence: {result['tier1']['confidence']:.2%}")
    
    if result['tier1'].get('alternatives'):
        print("  Alternatives:")
        for alt in result['tier1']['alternatives']:
            print(f"    - {alt['label']}: {alt['confidence']:.2%}")
    
    # Tier 2
    if result['tier2'] and not result['tier2'].get('message'):
        print("\n🔍 TIER 2 - Attribute Detection:")
        tier2 = result['tier2']
        for key, value in tier2.items():
            if key not in ['processing_time_ms'] and not key.endswith('_alternatives'):
                print(f"  {key.replace('_', ' ').title()}: {value}")
    
    # Metadata
    if result.get('metadata'):
        print("\n⏱️  Performance Metrics:")
        meta = result['metadata']
        if 'total_time_ms' in meta:
            print(f"  Processing Time: {meta['total_time_ms']:.0f}ms")
        if 'overall_confidence' in meta:
            print(f"  Overall Confidence: {meta['overall_confidence']:.2%}")

display_result(result)

## 6. Test Tier 1 Only (Category Detection)

In [ ]:
# Create a standalone Tier 1 detector
tier1_detector = create_category_detector()

# Test on an image
tier1_result = tier1_detector.predict(image_path, top_k=5)

print("Top 5 category predictions:")
print(f"1. {tier1_result['category']} ({tier1_result['confidence']:.2%})")
for i, alt in enumerate(tier1_result['alternatives'], 2):
    print(f"{i}. {alt['label']} ({alt['confidence']:.2%})")

## 7. Test Electronics Detector (Tier 2)

In [ ]:
# Create standalone electronics detector
electronics_detector = create_electronics_detector()

# Test on an electronics image
# (Replace with actual electronics image for better results)
electronics_result = electronics_detector.predict(image_path, detect_all=True)

print("Electronics detection results:")
print(json.dumps(electronics_result, indent=2))

## 8. Batch Processing

Process multiple images at once.

In [ ]:
# List of images to process
image_paths = [
    '../data/raw/test_image.jpg',
    # Add more image paths here
]

# Process batch
batch_results = pipeline.predict_batch(image_paths, return_metadata=False)

# Display summary
print("Batch processing results:")
for i, result in enumerate(batch_results, 1):
    if 'error' not in result:
        category = result['tier1']['category']
        confidence = result['tier1']['confidence']
        print(f"{i}. {category} ({confidence:.1%})")
    else:
        print(f"{i}. ERROR: {result['error']}")

## 9. Adjust Confidence Threshold

In [ ]:
# Lower threshold to allow more Tier 2 predictions
pipeline.set_confidence_threshold(0.3)

# Or higher threshold for more conservative Tier 2 routing
# pipeline.set_confidence_threshold(0.7)

print("Confidence threshold updated!")

## 10. Next Steps

- **Collect real data**: Gather marketplace images for testing
- **Add more Tier 2 detectors**: Implement specialized detectors for furniture, clothing, vehicles, etc.
- **Fine-tune models**: Train on marketplace-specific data
- **Evaluate performance**: Measure accuracy on test set
- **Deploy**: Create API endpoint for production use

## Resources

- **Source code**: `../src/`
- **Documentation**: `../README.md`
- **Tests**: `../tests/`
- **Demo script**: `../demo.py`